1. Extract data from Crypto API using requests
2. Load raw data into dataframe
3. Clean and transform raw data
4. Load into MySQL using sqlalchemy and mysql-connector

In [32]:
import pandas as pd
import numpy as np
from logger import logger
from extract import extract
from config import *


In [33]:
data = extract(URL)

In [34]:
df_raw = pd.DataFrame(data)
df_raw.head()

""


In [7]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 26 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                250 non-null    object 
 1   symbol                            250 non-null    object 
 2   name                              250 non-null    object 
 3   image                             250 non-null    object 
 4   current_price                     250 non-null    float64
 5   market_cap                        250 non-null    int64  
 6   market_cap_rank                   250 non-null    int64  
 7   fully_diluted_valuation           250 non-null    int64  
 8   total_volume                      250 non-null    float64
 9   high_24h                          250 non-null    float64
 10  low_24h                           250 non-null    float64
 11  price_change_24h                  250 non-null    float64
 12  price_ch

In [21]:
# Remove duplicate coins
df.drop_duplicates(subset=["id"],inplace=True)

numeric_columns = [
    "current_price",
    "market_cap",
    "market_cap_rank",
    "total_volume",
    "high_24h",
    "low_24h",
    "price_change_24h",
    "price_change_percentage_24h",
    "circulating_supply"
]

# convert numeric columns
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column],
         errors="coerce"                      
    )

# convert date columns
df["last_updated"] = pd.to_datetime(
    df["last_updated"],
    errors='coerce'
)

# Capitalize symbol
df["symbol"] = df["symbol"].str.upper()

# Rename columns
df.rename(columns={
    "current_price": "price",
    "total_volume": "volume",
    "price_change_percentage_24h": "price_change_pct_24h"
}, inplace=True)



C:\Users\kvsur\AppData\Local\Temp\ipykernel_21636\2849146085.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(subset=["id"],inplace=True)


KeyError: 'current_price'

In [24]:
# Remove spaces
df["name"] = df["name"].str.strip()

# Indicator

df["indicator"] = np.where(
    df["price_change_pct_24h"]>2,"Buy",
    np.where(
        df["price_change_pct_24h"]>=0,"Hold","Sell"
    )
)

C:\Users\kvsur\AppData\Local\Temp\ipykernel_21636\1879774854.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["name"] = df["name"].str.strip()
C:\Users\kvsur\AppData\Local\Temp\ipykernel_21636\1879774854.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["indicator"] = np.where(


In [27]:
df.head(25)

,id,symbol,name,price,market_cap,market_cap_rank,volume,high_24h,low_24h,price_change_24h,price_change_pct_24h,circulating_supply,last_updated,indicator
0,bitcoin,BTC,Bitcoin,83575.000000,1679092513073,1,2.804432e+10,84486.000000,82911.000000,-613.334318,-0.71300,2.009091e+07,2026-09-30 09:50:30+00:00,Sell
1,ethereum,ETH,Ethereum,2688.640000,328248153981,2,1.337445e+10,2742.680000,2657.670000,-27.879446,-1.06285,1.220929e+08,2026-09-30 09:50:30+00:00,Sell
2,tether,USDT,Tether,0.999582,183797056462,3,5.591436e+10,0.999907,0.999573,-0.000126,-0.01537,1.838706e+11,2026-09-30 09:50:30+00:00,Sell
3,binancecoin,BNB,BNB,764.220000,101774813949,4,8.427698e+08,767.010000,750.630000,-0.537934,-0.06649,1.331595e+08,2026-09-30 09:50:30+00:00,Sell
4,ripple,XRP,XRP,1.510000,95409871217,5,3.820677e+09,1.560000,1.480000,0.001648,0.13910,6.309298e+10,2026-09-30 09:50:30+00:00,Hold
5,usd-coin,USDC,USDC,0.999791,74339971306,6,1.654203e+10,1.000000,0.999773,-0.000075,-0.01094,7.435409e+10,2026-09-30 09:50:30+00:00,Sell
6,solana,SOL,Solana,119.280000,70131598806,7,3.573953e+09,121.580000,117.460000,-0.548364,-0.42530,5.880060e+08,2026-09-30 09:50:30+00:00,Sell
7,tron,TRX,TRON,0.338323,32132643340,8,3.176929e+08,0.338471,0.333923,0.002757,0.80984,9.497573e+10,2026-09-30 09:50:30+00:00,Hold
8,figure-heloc,FIGR_HELOC,Figure Heloc,1.030000,23957411220,9,5.451094e+08,1.030000,1.001000,0.027431,NaN,2.326520e+10,2026-09-30 09:50:30+00:00,Sell
9,zcash,ZEC,Zcash,1407.970000,23874953748,10,9.382860e+08,1458.280000,1382.170000,-14.278739,-0.88573,1.696008e+07,2026-09-30 09:50:30+00:00,Sell


In [ ]:
columns = [
    "id",
    "symbol",
    "name",
    "current_price",
    "market_cap",
    "market_cap_rank",
    "total_volume",
    "high_24h",
    "low_24h",
    "price_change_24h",
    "price_change_percentage_24h",
    "circulating_supply",
    "last_updated"
]

In [3]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

API_KEY = os.getenv("API_KEY")
print(API_KEY)

CG-RsZ22p2fQ24zgZ5drMU89pnc
